# Task Manager: Secure Multi-User CLI with Authentication & Persistence

### Problem Statement
Build a command-line Task Manager that supports multi-user authentication.
Each user registers with a username/password, logs in, and can then create,
view, update, and delete **their own** tasks — with no user able to see
another user's data. Data must persist across sessions using file storage.

### Why this project matters
Before a model ever touches production, the surrounding system needs to get
the fundamentals right: secure credential handling, clean data modeling,
safe input handling, and durable persistence. This project is a deliberate,
scoped exercise in those fundamentals, built the way I'd want a real pull
request to look — with the reasoning behind each design decision made
explicit rather than left implicit.

*(Built while deepening my AI/ML engineering skill set through the UT Dallas
x Fullstack Academy AI & Machine Learning program.)*

### Skills Exhibited
| Category | Skill |
|---|---|
| **Security fundamentals** | Password hashing + salting (never storing plaintext credentials) |
| **Data structures** | Nested dictionaries for per-user task isolation |
| **File I/O & persistence** | JSON read/write for durable storage across sessions |
| **Defensive programming** | `try/except` blocks around all user input and file operations |
| **Software design** | Single-responsibility functions, clear separation of auth vs. task logic |
| **UX for CLI tools** | Menu-driven interface with input validation and clear feedback |
| **Documentation** | Docstrings + inline reasoning (this notebook itself) |

### What I'd do differently at scale
In a production system I would swap SHA-256 salted hashing for **bcrypt/argon2**
(purpose-built, slow-by-design hashing algorithms resistant to brute force),
replace the JSON files with a proper database (Postgres/DynamoDB) to handle
concurrent access safely, and add structured logging instead of print statements.
I call these out explicitly below rather than pretending a CLI toy project
is production-grade — that distinction matters when discussing this project
in interviews.


## 1. Setup & Persistence Layer

We store data in two JSON files: `users.json` (credentials) and `tasks.json`
(tasks, keyed by username). JSON gives us human-readable, dependency-free
persistence that's perfect for a project this size — a real production system
would swap this layer out for a database without changing the function
signatures above it. That separation (storage logic isolated in its own
functions) is exactly why it's easy to swap later.


In [1]:
import json
import os
import hashlib
import secrets

USERS_FILE = "users.json"
TASKS_FILE = "tasks.json"


def load_json(filepath):
    """Load a JSON file into a dict. Returns an empty dict if the file
    doesn't exist yet (first run) or is corrupted (defensive fallback)."""
    if not os.path.exists(filepath):
        return {}
    try:
        with open(filepath, "r") as f:
            return json.load(f)
    except (json.JSONDecodeError, IOError):
        print(f"Warning: could not read {filepath}, starting fresh.")
        return {}


def save_json(filepath, data):
    """Persist a dict to disk as JSON."""
    with open(filepath, "w") as f:
        json.dump(data, f, indent=2)


## 2. Authentication — Registration & Login

**Security design decisions (and why):**
- Passwords are never stored in plaintext.
- Each user gets a unique, randomly generated **salt** (`secrets.token_hex`),
  which is combined with their password before hashing. This defeats
  precomputed "rainbow table" attacks and ensures two users with the same
  password produce different hashes.
- `secrets` (not `random`) is used for the salt because it's cryptographically
  secure — a detail that matters even in a learning project.
- Usernames are enforced unique at registration time.

> **Interview-ready callout:** SHA-256 is fast, which is actually a weakness
> for password hashing (fast = brute-forceable at scale). In production I'd
> use `bcrypt` or `argon2`, which are intentionally slow and have built-in
> salting. I'm using `hashlib` + manual salting here to demonstrate I
> understand *why* salting exists at the algorithm level, not just how to
> call a library.


In [2]:
def hash_password(password: str, salt: str) -> str:
    """Combine password + salt and hash with SHA-256."""
    return hashlib.sha256((salt + password).encode()).hexdigest()


def register_user(username: str, password: str) -> bool:
    """
    Register a new user. Returns True on success, False if the
    username is already taken.
    """
    users = load_json(USERS_FILE)

    if username in users:
        print(f"Username '{username}' is already taken. Please choose another.")
        return False

    salt = secrets.token_hex(16)
    hashed = hash_password(password, salt)

    users[username] = {"salt": salt, "password_hash": hashed}
    save_json(USERS_FILE, users)

    # Initialize an empty task list for the new user
    tasks = load_json(TASKS_FILE)
    tasks[username] = []
    save_json(TASKS_FILE, tasks)

    print(f"User '{username}' registered successfully.")
    return True


def login_user(username: str, password: str) -> bool:
    """
    Validate credentials against stored (salted, hashed) password.
    Returns True if credentials are valid, False otherwise.
    """
    users = load_json(USERS_FILE)

    if username not in users:
        print("Username not found.")
        return False

    salt = users[username]["salt"]
    hashed_attempt = hash_password(password, salt)

    if hashed_attempt == users[username]["password_hash"]:
        print(f"Welcome back, {username}!")
        return True
    else:
        print("Incorrect password.")
        return False


## 3. Task Management (Per-User Isolation)

Tasks are stored as `{username: [list of task dicts]}` — a nested structure
that guarantees each user's data is logically isolated inside a single file.
Each task has a unique ID (auto-incremented per user), a description, and a
status (`Pending` / `Completed`).

This is the same pattern used at larger scale in multi-tenant SaaS systems —
row-level or namespace-level isolation by user/tenant ID — just implemented
here with dictionaries instead of database rows.


In [3]:
def _next_task_id(user_tasks: list) -> int:
    """Generate the next unique task ID for a user (max existing ID + 1)."""
    if not user_tasks:
        return 1
    return max(task["id"] for task in user_tasks) + 1


def add_task(username: str, description: str) -> dict:
    """Add a new Pending task for the given user and persist it."""
    tasks = load_json(TASKS_FILE)
    user_tasks = tasks.get(username, [])

    new_task = {
        "id": _next_task_id(user_tasks),
        "description": description,
        "status": "Pending",
    }
    user_tasks.append(new_task)
    tasks[username] = user_tasks
    save_json(TASKS_FILE, tasks)

    print(f"Task added: [{new_task['id']}] {description}")
    return new_task


def view_tasks(username: str) -> list:
    """Print and return all tasks belonging to the given user."""
    tasks = load_json(TASKS_FILE)
    user_tasks = tasks.get(username, [])

    if not user_tasks:
        print("No tasks found. Add one to get started!")
        return []

    print(f"\n--- Tasks for {username} ---")
    for task in user_tasks:
        print(f"[{task['id']}] {task['description']} - {task['status']}")
    print("-" * 28)
    return user_tasks


def complete_task(username: str, task_id: int) -> bool:
    """Mark a task as Completed by ID. Returns True if found and updated."""
    tasks = load_json(TASKS_FILE)
    user_tasks = tasks.get(username, [])

    for task in user_tasks:
        if task["id"] == task_id:
            task["status"] = "Completed"
            tasks[username] = user_tasks
            save_json(TASKS_FILE, tasks)
            print(f"Task {task_id} marked as Completed.")
            return True

    print(f"Task ID {task_id} not found.")
    return False


def delete_task(username: str, task_id: int) -> bool:
    """Delete a task by ID. Returns True if found and removed."""
    tasks = load_json(TASKS_FILE)
    user_tasks = tasks.get(username, [])

    for task in user_tasks:
        if task["id"] == task_id:
            user_tasks.remove(task)
            tasks[username] = user_tasks
            save_json(TASKS_FILE, tasks)
            print(f"Task {task_id} deleted.")
            return True

    print(f"Task ID {task_id} not found.")
    return False


## 4. Interactive Menu-Driven Interface

The `main()` function ties everything together into a login/register flow
followed by a task-management loop. Every user input is wrapped in
`try/except` — a deliberate defensive-programming choice so that bad input
(e.g., typing "abc" where a task ID/int is expected) never crashes the
program. Instead, the user gets a friendly message and another chance.

> **Note on running this cell:** `main()` uses blocking `input()` calls,
> which work fine in a real terminal (`python task_manager.py`) or in an
> interactive Jupyter session, but will not execute inside a static GitHub
> preview of this notebook. For that reason, **Section 5 below runs a full
> non-interactive demo** so the logic and output are visible without needing
> a live terminal. Feel free to run `main()` yourself in Jupyter or from the
> command line to try the real interactive version.


In [4]:
def main():
    print("=== Welcome to Task Manager ===")

    while True:
        choice = input("Do you want to (L)ogin or (R)egister? [L/R]: ").strip().lower()

        if choice == "r":
            username = input("Choose a username: ").strip()
            password = input("Choose a password: ").strip()
            if register_user(username, password):
                break
        elif choice == "l":
            username = input("Username: ").strip()
            password = input("Password: ").strip()
            if login_user(username, password):
                break
        else:
            print("Invalid choice. Please enter 'L' or 'R'.")

    # Task management loop
    while True:
        print("\n1. Add Task\n2. View Tasks\n3. Mark Task Completed\n4. Delete Task\n5. Logout")
        try:
            option = input("Choose an option (1-5): ").strip()

            if option == "1":
                description = input("Task description: ").strip()
                add_task(username, description)

            elif option == "2":
                view_tasks(username)

            elif option == "3":
                task_id = int(input("Task ID to mark completed: ").strip())
                complete_task(username, task_id)

            elif option == "4":
                task_id = int(input("Task ID to delete: ").strip())
                delete_task(username, task_id)

            elif option == "5":
                print(f"Goodbye, {username}!")
                break

            else:
                print("Invalid option. Please choose 1-5.")

        except ValueError:
            print("That doesn't look like a valid number. Please try again.")
        except Exception as e:
            print(f"Unexpected error: {e}. Please try again.")


# To run the real interactive version, uncomment the line below
# and run this cell in a live Jupyter kernel or as a .py script:
# main()


## 5. Non-Interactive Demo (Proof of Functionality)

This section calls the same functions `main()` uses, but with hard-coded
inputs instead of `input()`, so the full flow — register, login, add tasks,
complete a task, delete a task, view final state — is demonstrated with
visible output directly in this notebook (and on GitHub) without needing a
live terminal.


In [5]:
# Clean slate for the demo (safe to skip if you want data to persist)
for f in (USERS_FILE, TASKS_FILE):
    if os.path.exists(f):
        os.remove(f)

print(">>> Registering a new user")
register_user("feyisayo_demo", "SecurePass123")

print("\n>>> Attempting duplicate registration (should fail gracefully)")
register_user("feyisayo_demo", "AnotherPassword")

print("\n>>> Logging in with correct credentials")
login_user("feyisayo_demo", "SecurePass123")

print("\n>>> Logging in with WRONG password (should fail gracefully)")
login_user("feyisayo_demo", "wrongpassword")


>>> Registering a new user
User 'feyisayo_demo' registered successfully.

>>> Attempting duplicate registration (should fail gracefully)
Username 'feyisayo_demo' is already taken. Please choose another.

>>> Logging in with correct credentials
Welcome back, feyisayo_demo!

>>> Logging in with WRONG password (should fail gracefully)
Incorrect password.


False

In [6]:
print(">>> Adding tasks")
add_task("feyisayo_demo", "Finish Unit 1 programming refresher")
add_task("feyisayo_demo", "Push Task Manager project to GitHub")
add_task("feyisayo_demo", "Write LinkedIn post about the project")

print()
view_tasks("feyisayo_demo")


>>> Adding tasks
Task added: [1] Finish Unit 1 programming refresher
Task added: [2] Push Task Manager project to GitHub
Task added: [3] Write LinkedIn post about the project


--- Tasks for feyisayo_demo ---
[1] Finish Unit 1 programming refresher - Pending
[2] Push Task Manager project to GitHub - Pending
[3] Write LinkedIn post about the project - Pending
----------------------------


[{'id': 1,
  'description': 'Finish Unit 1 programming refresher',
  'status': 'Pending'},
 {'id': 2,
  'description': 'Push Task Manager project to GitHub',
  'status': 'Pending'},
 {'id': 3,
  'description': 'Write LinkedIn post about the project',
  'status': 'Pending'}]

In [7]:
print(">>> Marking task 1 as completed")
complete_task("feyisayo_demo", 1)

print("\n>>> Deleting task 2")
delete_task("feyisayo_demo", 2)

print("\n>>> Final task state")
view_tasks("feyisayo_demo")

print("\n>>> Attempting to complete a task that doesn't exist (should fail gracefully)")
complete_task("feyisayo_demo", 999)


>>> Marking task 1 as completed
Task 1 marked as Completed.

>>> Deleting task 2
Task 2 deleted.

>>> Final task state

--- Tasks for feyisayo_demo ---
[1] Finish Unit 1 programming refresher - Completed
[3] Write LinkedIn post about the project - Pending
----------------------------

>>> Attempting to complete a task that doesn't exist (should fail gracefully)
Task ID 999 not found.


False

## 6. Reflection — Skills Summary

This project was intentionally scoped as a foundations exercise, but it maps
directly to competencies I'll carry into every later unit of this bootcamp
and into an AI/ML Engineer role:

- **Secure-by-default thinking** — salting/hashing credentials, never
  storing secrets in plaintext, and validating input before trusting it.
  This mindset carries directly into securing API keys, model endpoints, and
  user data in production ML systems.
- **Clean data modeling** — designing a nested, per-user data structure
  up front prevented data leakage between users, the same principle behind
  multi-tenant data isolation in real systems.
- **Defensive programming** — wrapping every user-facing input in error
  handling so the system degrades gracefully instead of crashing, a habit
  that matters just as much in an inference pipeline as in a CLI tool.
- **Separation of concerns** — persistence, authentication, and task logic
  each live in their own functions, making the system easy to test, extend,
  or swap out pieces of (e.g., replacing JSON with a real database) without
  a rewrite.

**Next:** extending this same rigor into full data science and ML
workflows — exploratory analysis, feature engineering, and model
development held to the same production-quality standard.
